In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[0:10]
OPENAI_API_KEY

'sk-proj-GG'

In [2]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatMessagePromptTemplate, ChatPromptTemplate

In [3]:
chat = ChatOpenAI()

In [4]:
chat.invoke("호날두vs메시 하나만 골라 너가")

AIMessage(content=' 호날두를 고를게.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 24, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DeboGSyOXwAzZjydSyPSjgzTIrCSu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b15-25d1-7a33-b082-ed85b612a28f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 11, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## temperature(창의력)
- 0.0 ~ 0.3: 결정론 - 요약, 추출, 일관, 정확성
- 0.5 ~ 0.7: 균형적 - 적당히 자연스러움, 유연성
- 0.8 ~ 1.0+: 무작위 - 창의적이고 예측 불가

In [5]:
chat = ChatOpenAI(temperature=0)

result = chat.invoke("하늘의 색은?")

result.content

'하늘의 색은 파란색이다.'

In [6]:
chat = ChatOpenAI(temperature=0.5)

result = chat.invoke("하늘의 색은?")

result.content

'하늘의 색은 파란색이다.'

In [9]:
chat = ChatOpenAI(temperature=1.0)

result = chat.invoke("하늘의 색은?")

result.content

'하늘의 색은 파란색이다.'

In [10]:
chat.invoke("아스날vspsg 누가 이길거 같아 한 팀 딱 골라")

AIMessage(content='PSG가 이길 것 같습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 32, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DebtzMnHQMu5qMS6l0KxRPEHo8rwc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b1a-9177-7e82-bb61-3c77b43beb7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 10, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [17]:
class NewLineOutputParser(BaseOutputParser):
    # 반드시 parse
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip("-123456789. ").strip() for line in lines]

In [18]:
newline_parser = NewLineOutputParser()

In [19]:
newline_parser.parse("""- 1. 햄버거\n- 2. 떡볶이\n- 3. 치킨""")

['햄버거', '떡볶이', '치킨']

In [20]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요.
    """),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_length=5,
    question="AI를 잘할라면 어떤거부터 공부해야해?"
)

## 체인(Chain 생성)
- "|": 파이프 연산자로 체인을 만든다.

In [21]:
# List
first_chain = template | chat | NewLineOutputParser()

In [23]:
chain_result = first_chain.invoke({
    "max_length":5,
    "question": "AI를 잘할라면 어떤거부터 공부해야해?"
})

In [24]:
chain_result

['AI를 잘 하기 위해서는 다음과 같은 순서로 공부하는 것을 추천합니다:',
 '',
 'Python 프로그래밍 언어',
 '데이터 구조와 알고리즘',
 '선형대수학',
 '통계학',
 '머신 러닝',
 '딥러닝',
 '',
 '이외에도 AI 분야에서 사용되는 다양한 기술과 알고리즘에 대해 학습하고, 프로젝트를 경험해보는 것도 중요합니다.']

In [25]:
# RunnableSequence
# 이전 단계의 출력이 다음 단계의 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 1. 템플릿 생성

In [32]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행 가이드입니다.
        사람들이 좋아하는 여행 장소를 많이 알고 있습니다.
        설명없이 지역 명소 이름만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 여행 장소 추천해줘!
    """)
])

In [33]:
second_chain = template | chat

In [34]:
trip_result = second_chain.invoke({
    "max_length": 3,
    "place": "부산"
})

In [35]:
trip_result.content

'1. 해운대해수욕장\n2. 부산 국립해양박물관\n3. 태종대'

## 실습

In [39]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 음식 추천 가이드입니다.
        사람들이 좋아하는 음식을 많이 알고 있습니다.
        설명없이 저녁 메뉴만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 음식 메뉴 추천해줘!
    """)
])

In [40]:
third_chain = template | chat

In [49]:
menu_result = third_chain.invoke({
    "max_length": 3,
    "place": "우주식"
})

In [50]:
menu_result.content

'1. 갈색 다람쥐 스튜\n2. 운명의 별 파스타\n3. 외계 피자'